# Universidad Externado de Colombia · Facultad de Finanzas, Gobierno y Relaciones Internacionales
## Laboratorio Cuantitativo de Valoración de Activos y Gestión de Riesgo (Kibo)
**Materia: Valoración de Activos (DEFI 2)**

Este notebook reproduce paso a paso el 100% de los requerimientos analíticos del taller oficial:
1. **Datos & Universo de Activos**: Ingesta de 10 acciones en USD de 6 sectores, partición 504 calibración / 252 prueba, divisor muestral $n-1$.
2. **Diagnóstico del Mercado (Efecto Manada CCK 2000)**: Desviación absoluta transversal $CSAD_t$ y regresión cuadrática de contagio.
3. **Proyecciones Estocásticas**: Calibración y simulación Monte Carlo de GBM, Merton Jump, Ornstein-Uhlenbeck y fBm con exponente de Hurst ($H$).
4. **Opciones & Árbol Binomial CRR**: 9 contratos candidatos (cuantiles $Q_{25}, Q_{50}, Q_{75}$), valoración CRR vs Black-Scholes, test de control obligatorio ($10.4506$) y árbol de 3 pasos.
5. **Asignación de Capital & Presupuesto de Riesgo**: Comparación de 1/N, Inverso de VaR y Mínimo VaR (SLSQP) con límites $2\% \le w_i \le 20\%$ y sectorial $\le 35\%$, factor $f$, acciones enteras y efectivo residual.
6. **Medición de Riesgo**: 3 estimadores de VaR al 99% a 10 días (Paramétrico Normal, Histórico Empírico, Monte Carlo 20.000 trayectorias Cholesky) y 3 escenarios de estrés.
7. **Auditoría Fuera de Muestra**: Evolución del patrimonio de USD 100.000 en 252 sesiones, costo de corretaje de 10 bps y Test de Kupiec (POF).
8. **Evaluación Predictiva Rolling del GBM**: Ventana móvil de 252 sesiones pronosticando a $H=1$ día para las 10 acciones con métricas RMSE y MAPE vs caminata aleatoria.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats, optimize

from src.data_engine import download_market_data, DEFAULT_TICKERS, SECTOR_MAP, get_default_portfolio, get_library_versions
from src.herding_engine import run_herding_analysis
from src.stochastic_engine import (
    calibrate_gbm, simulate_gbm,
    calibrate_merton_jump, simulate_merton_jump,
    calibrate_ornstein_uhlenbeck, simulate_ornstein_uhlenbeck
)
from src.fractional_engine import estimate_hurst_rs, simulate_fractional_brownian_motion
from src.options_engine import (
    get_option_candidates, crr_price, contract_probabilities,
    generate_crr_3step_tree, verify_crr_control_benchmark, black_scholes_price
)
from src.portfolio_engine import compute_portfolio_allocation
from src.risk_engine import compute_var_comparison_and_stress, run_out_of_sample_portfolio_audit
from src.rolling_engine import run_gbm_rolling_evaluation, run_multi_asset_rolling_table

print('✓ Todos los módulos cuantitativos de Kibo importados exitosamente.')
print('Versiones de librerías registradas:')
for k, v in get_library_versions().items():
    print(f'  - {k}: {v}')


### 1. Ingesta y Calibración de Datos (10 Acciones · 6 Sectores)
Descarga de más de 3 años diarios vía yfinance con `auto_adjust=True` para retornos continuos y spot de cierre sin ajustar. Auditoría de 756+ sesiones comunes y cálculo de parámetros oficiales (Paso 3).


In [ ]:
prices_df, returns_df, stats_dict = download_market_data(DEFAULT_TICKERS, years=3)
print(f'Sesiones comunes completas: {len(prices_df)} (Mínimo exigido: 756)')
print(f'Rango temporal: {prices_df.index[0].date()} a {prices_df.index[-1].date()}')
print('\nTabla de Parámetros Calibrados Oficiales (Paso 3 del Taller):')
display(stats_dict['parameters_table'])


### 2. Test Econométrico de Comportamiento en Manada (Herding CCK 2000)
Estimación de la regresión cuadrática:
$$CSAD_t = \alpha + \gamma_1 |R_{m,t}| + \gamma_2 R_{m,t}^2 + \varepsilon_t$$


In [ ]:
herding_res = run_herding_analysis(returns_df)
print(f'Veredicto Oficial: {herding_res["verdict_badge"]} -> {herding_res["verdict"]}')
print(f'Coeficiente gamma_2: {herding_res["gamma2"]:.5f} (t = {herding_res["gamma2_t"]:.2f}, p-valor = {herding_res["gamma2_p"]:.4f})')
print(f'Bondad de ajuste R^2: {herding_res["r_squared"]:.2%}')
print(f'Diagnóstico de Asimetría: {herding_res["asym_insight"]}')


### 3. Proyecciones Estocásticas: GBM y Competencia Multimodelo (Paso 4)
Simulación Monte Carlo bajo medida física $\mathbb{P}$ con semilla fija 42, contraste de medias y exponente de Hurst ($H$).


In [ ]:
asset = 'AAPL'
spot_asset = float(stats_dict[asset]['raw_spot'])
params_gbm = calibrate_gbm(returns_df[asset])
sim_gbm = simulate_gbm(spot_asset, params_gbm, horizon_days=10, n_paths=5000, seed=42)

positivity_ok = np.all(sim_gbm.paths > 0)
diff_mean = sim_gbm.simulated_mean - sim_gbm.theoretical_mean

print(f'=== RESULTADOS SIMULACIÓN GBM ({asset}) ===')
print(f'Spot Inicial S_0: ${spot_asset:.2f}')
print(f'Media Simulada S_barra_T: ${sim_gbm.simulated_mean:.2f} (Error estándar: ±${sim_gbm.std_error:.2f})')
print(f'Media Teórica S_0*exp(aT): ${sim_gbm.theoretical_mean:.2f} (Diferencia: ${diff_mean:+.2f})')
print(f'Chequeo de Positividad (S_t > 0): {"✓ Cumplido" if positivity_ok else "X Violado"}')

# Memoria fractal con Exponente de Hurst
hurst_res = estimate_hurst_rs(prices_df[asset].values)
print(f'Exponente de Hurst H (Análisis R/S): {hurst_res["hurst"]:.4f} -> {hurst_res["regime"]}')


### 4. Cobertura con Opciones Reales & Árbol Binomial CRR (Pasos 5 a 9)
Valoración binomial CRR (Europea/Americana), validación del benchmark obligatorio del PDF ($10.4506$) y árbol de 3 pasos.


In [ ]:
# 1. Test de Control Oficial del Taller
bench = verify_crr_control_benchmark()
print(f'=== TEST DE CONTROL OFICIAL DEL TALLER (Call Europea S0=100, K=100, T=1, r=0.05, sigma=0.20) ===')
print(f'Valor Teórico Exigido: USD ${bench["target"]:.4f}')
print(f'Black-Scholes Analítico: USD ${bench["black_scholes_exact"]:.4f}')
print(f'Árbol CRR (200 pasos): USD ${bench["crr_200"]:.4f} (Diferencia: ${bench["difference"]:.4f})')
print(f'Árbol CRR (500 pasos): USD ${bench["crr_500"]:.4f}')
print(f'Veredicto de Control: {"✓ Aprobado con Éxito" if bench["passed"] else "X Discrepancia"}\n')

# 2. Valoración de Contrato Seleccionado
opt_ticker = 'AAPL'
t_exp = 91 / 365.0
k_strike = round(spot_asset, 0)
val_call_euro = crr_price(spot_asset, k_strike, t_exp, rate=0.04, dividend_yield=0.0, sigma=stats_dict[opt_ticker]['sigma_annual'], steps=200, option_type='Call', style='Europea')
val_call_amer = crr_price(spot_asset, k_strike, t_exp, rate=0.04, dividend_yield=0.0, sigma=stats_dict[opt_ticker]['sigma_annual'], steps=200, option_type='Call', style='Americana')

print(f'=== VALORACIÓN DE OPCIÓN SOBRE {opt_ticker} (Strike ${k_strike:.2f}, Vencimiento 91d) ===')
print(f'Prima CRR Europea: USD ${val_call_euro["price"]:.2f}')
print(f'Prima CRR Americana: USD ${val_call_amer["price"]:.2f} (Teorema de Merton: coincide por no haber dividendo)')
print(f'Black-Scholes Benchmark: USD ${val_call_euro["black_scholes"]:.2f}')

# 3. Sub-árbol de 3 pasos
fig_tree, df_tree_nodes = generate_crr_3step_tree(spot_asset, k_strike, t_exp, rate=0.04, dividend_yield=0.0, sigma=stats_dict[opt_ticker]['sigma_annual'], option_type='Call', style='Europea')
print('\nTabla de Nodos del Árbol Binomial de 3 Pasos (Paso 8):')
display(df_tree_nodes)


### 5. Asignación Óptima de Capital y Presupuesto de Riesgo (Pasos 11 y 12)
Optimización de Mínimo VaR restringido (SLSQP):
$$\min_{w} \quad w^T \Sigma_R w \quad \text{s.a.} \quad \sum w_i = 1, \quad 0.02 \le w_i \le 0.20, \quad \sum_{i \in \text{Sector}} w_i \le 0.35$$
Cálculo de acciones enteras $A_i = \lfloor B w_i / S_{i,0} \rfloor$ y factor de ajuste por presupuesto de riesgo $f = \min\{1, \bar{V}/V_{\text{riesgo}}\}$. 


In [ ]:
capital_base = 100000.0
risk_budget_v = 7500.0
portfolio_df = get_default_portfolio()

alloc_res = compute_portfolio_allocation(
    capital=capital_base,
    portfolio_df=portfolio_df,
    returns_df=returns_df,
    confidence=99.0,
    horizon_days=10,
    prices_df=prices_df,
    risk_budget_v=risk_budget_v
)

print(f'Capital Base: USD ${capital_base:,.2f}')
print(f'Presupuesto de Riesgo (V_bar): USD ${risk_budget_v:,.2f}')
print(f'Factor de Escala f: {alloc_res["f_scale"]:.4f}')
print(f'Total Invertido en Acciones: USD ${alloc_res["total_invertido_acciones"]:,.2f}')
print(f'Efectivo Libre de Riesgo: USD ${alloc_res["efectivo_total"]:,.2f}')

print('\nPlan de Compra y Ejecución de Acciones Enteras (Paso 11):')
display(alloc_res['allocation_df'])

print('\nCumplimiento de Límites Sectoriales (Cota <= 35%):')
display(alloc_res['sector_summary'])


### 6. Medición de Riesgo: 3 Estimadores de VaR al 99% y Pruebas de Estrés (Paso 10)
Comparación de VaR Paramétrico Normal, Histórico Empírico y Monte Carlo Multivariado con descomposición de Cholesky.


In [ ]:
df_alloc = alloc_res['allocation_df']
risk_res = compute_var_comparison_and_stress(
    capital=capital_base,
    weights=alloc_res['w_min_var'],
    returns_df=returns_df,
    sectors=df_alloc['Sector'].tolist(),
    tickers=df_alloc['Ticker'].tolist(),
    confidence=0.99,
    horizon_days=10,
    n_mc_sims=20000
)

print('=== COMPARATIVA DE 3 ESTIMADORES DE VaR Y CVaR AL 99% (10 DÍAS) ===')
display(risk_res['var_comparison_df'])

print('\n=== PRUEBAS DE ESTRÉS OFICIALES (STRESS TESTING) ===')
display(risk_res['stress_df'])


### 7. Auditoría Fuera de Muestra de la Cartera (252 Sesiones de Prueba)
Evaluación del desempeño en las 252 sesiones reservadas con deducción de 10 bps de costo de corretaje y contraste de excepciones mediante el Test de Cobertura de Kupiec ($LR_{POF} \sim \chi^2(1)$).


In [ ]:
oos_res = run_out_of_sample_portfolio_audit(
    prices_df=prices_df,
    returns_df=returns_df,
    weights=alloc_res['w_min_var'],
    tickers=df_alloc['Ticker'].tolist(),
    capital=capital_base,
    test_sessions=252,
    confidence=0.99,
    cost_bps=10.0
)

print('=== RESUMEN AUDITORÍA FUERA DE MUESTRA Y TEST DE KUPIEC ===')
display(oos_res['summary_table'])


### 8. Evaluación Predictiva Rolling del GBM (Paso 13 del PDF)
Ventana móvil de 252 sesiones de calibración diaria para reestimar $\hat{\mu}^{total}$ y $\hat{\sigma}$, avanzando 1 sesión diaria para pronosticar a $H=1$ día fuera de muestra. Cálculo de RMSE y MAPE para las 10 acciones vs caminata aleatoria.


In [ ]:
df_roll_table = run_multi_asset_rolling_table(prices_df, DEFAULT_TICKERS, window_train=252, horizon_H=1)
print('=== TABLA CONSOLIDADA DE ERRORES RMSE Y MAPE (H = 1 DÍA) ===')
display(df_roll_table)

# Inspección individual para AAPL
roll_aapl = run_gbm_rolling_evaluation(prices_df['AAPL'], 'AAPL', window_train=252, horizon_H=1)
print(f'\nMétricas individuales AAPL:')
print(f'RMSE Modelo: {roll_aapl["rmse_model"]:.4f} vs Naive: {roll_aapl["rmse_naive"]:.4f}')
print(f'MAPE Modelo: {roll_aapl["mape_model"]:.2%} vs Naive: {roll_aapl["mape_naive"]:.2%}')
